In [2]:
!conda install -c bioconda fastp -y

Channels:
 - bioconda
 - defaults
 - conda-forge
Platform: linux-64
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): repo.anaconda.com:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): repo.anaconda.com:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): conda.anaconda.org:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): repo.anaconda.com:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): conda.anaconda.org:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): repo.anaconda.com:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): conda.anaconda.org:443
| DEBUG:urllib3.connectionpool:https://repo.anaconda.com:443 "GET /pkgs/main/linux-64/repodata.json HTTP/1.1" 304 0
DEBUG:urllib3.connectionpool:https://repo.anaconda.com:443 "GET /pkgs/main/noarch/repodata.json HTTP/1.1" 304 0
DEBUG:urllib3.connectionpool:https://repo.anaconda.com:443 "GET /pkgs/r/noarch/repodata.json HTTP/1

In [4]:
import gzip

def create_mock_fastq(filename, read_id, sequence, quality):
    """Generates a tiny valid gzipped FASTQ file."""
    with gzip.open(filename, "wt") as f:
        # A valid FASTQ record requires exactly 4 lines per read
        f.write(f"@{read_id}\n")
        f.write(f"{sequence}\n")
        f.write("+\n")
        f.write(f"{quality}\n")
    print(f"Created mock file: {filename}")

# Generate matching Paired-End reads right inside your notebooks/ directory
create_mock_fastq("sample1_R1.fastq.gz", "read1/1", "ATCGATCGATCGATCGATCGATCGATCGATCG", "<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<")
create_mock_fastq("sample1_R2.fastq.gz", "read1/2", "CGATCGATCGATCGATCGATCGATCGATCGAT", "<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<")

Created mock file: sample1_R1.fastq.gz
Created mock file: sample1_R2.fastq.gz


In [9]:
import subprocess
import os
from pathlib import Path

def run_quality_control(input_forward, input_reverse, output_dir):
    """
    Wraps an external QC utility (fastp) to trim adapters and low-quality bases.
    """
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    # Extract sample name from the filename
    sample_name = Path(input_forward).name.split('_')[0]
    
    # Define outputs
    out_f = os.path.join(output_dir, f"{sample_name}_R1_clean.fastq.gz")
    out_r = os.path.join(output_dir, f"{sample_name}_R2_clean.fastq.gz")
    html_report = os.path.join(output_dir, f"{sample_name}_fastp.html")
    json_report = os.path.join(output_dir, f"{sample_name}_fastp.json")
    
    # Construct the command-line argument list exactly as you would type it in bash
    cmd = [
        "fastp",
        "-i", input_forward,
        "-I", input_reverse,
        "-o", out_f,
        "-O", out_r,
        "-h", html_report,
        "-j", json_report,
        "--detect_adapter_for_pe" # Auto-detect adapter sequences for paired-end reads
    ]
    
    print(f"Executing command: {' '.join(cmd)}")
    
    try:
        # Run the command and capture outputs
        result = subprocess.run(
            cmd, 
            stdout=subprocess.PIPE, 
            stderr=subprocess.PIPE, 
            text=True, 
            check=True
        )
        print("✅ QC Completed Successfully!")
        return True, result.stdout
        
    except FileNotFoundError:
        print("❌ Error: 'fastp' executable not found in your environment path.")
        print("💡 Solution: We need to install 'fastp' via bioconda.")
        return False, "Executable missing"
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Command failed with exit code {e.returncode}")
        print(f"Stderr output:\n{e.stderr}")
        return False, e.stderr

# --- TEST CELL RUN ---
# Let's test how it handles non-existent files/executables right now
run_quality_control("sample1_R1.fastq.gz", "sample1_R2.fastq.gz", "outputs/qc")

Executing command: fastp -i sample1_R1.fastq.gz -I sample1_R2.fastq.gz -o outputs/qc/sample1_R1_clean.fastq.gz -O outputs/qc/sample1_R2_clean.fastq.gz -h outputs/qc/sample1_fastp.html -j outputs/qc/sample1_fastp.json --detect_adapter_for_pe
✅ QC Completed Successfully!


(True, '')